<a href="https://colab.research.google.com/github/ibrahimymhafez/flyrank-ibrahim/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimymhafez/flyrank-ibrahim/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

data_path = 'https://raw.github.com/ibrahimymhafez/flyrank-ibrahim/main/data/raw/content_refresh_anonymized.csv'
df_raw = pd.read_csv(data_path)

df_month = df_raw.head(5000).copy()
df_month

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,content_36cef655bad7,client_bbb965ab0c,0.0,0.00,LOW,0.00,keyword article,informational,2668.0,19153.0,...,15000-25000,0.00,28.1,0.00,0.00,0.0,moderate,page_3_5,down,-31.6
4996,content_1787097114c8,client_3fdba35f04,10.0,0.02,LOW,0.00,keyword article,commercial,2305.0,14100.0,...,8000-15000,0.06,27.1,14.29,18.18,0.0,good,page_3_5,down,-73.6
4997,content_fcde068b12d7,client_f369cb89fc,10.0,1.00,HIGH,1.25,keyword article,transactional,2789.0,16372.0,...,15000-25000,0.36,38.1,0.00,18.18,0.0,moderate,page_3_5,up,98.1
4998,content_5d5d0256fabb,client_624b60c58c,NaN,NaN,NaN,NaN,feedly article,NaN,1110.0,7627.0,...,<8000,0.00,6.5,0.00,50.00,0.0,low,page_1,down,-75.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
context = ['content_id']
features = ['search_volume', 'clicks_90d', 'avg_position', 'impressions_90d', 'content_age_days']
excluded = ['clicks_next_30d']
label = 'needs_refresh'

if 'engagement_rate' in df_month.columns:
  threshold = df_month['engagement_rate'].quantile(0.25)
  df_month[label] = (df_month['engagement_rate'] > threshold).astype(int)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
if 'content_id' in df_month.columns:
  is_unique = df_month['content_id'].is_unique
  print(f"Grain Check: content_id is unique per row? {is_unique}")

row_count = df_month.shape[0]
print(f"Row Count: {row_count}")

missing_values = df_month.isnull().sum()
print(f"Missing Values: {missing_values.count()}")

df_feature_frame = df_month[context + features + [label]].copy()
survivors = df_feature_frame.dropna()
survival_count = len(survivors)
print(f"Availability: {survival_count} rows out of {row_count} survive the complete data filter.")

# Define 'windows' based on content_age_days
# For example, let's categorize content into age groups
def create_age_window(days):
    if days <= 30:
        return '0-30 Days (New)'
    elif days <= 90:
        return '31-90 Days (Recent)'
    elif days <= 180:
        return '91-180 Days (Medium)'
    else:
        return '180+ Days (Old)'

df_month['content_age_window'] = df_month['content_age_days'].apply(create_age_window)

print("\nContent Age Window Distribution:")
print(df_month['content_age_window'].value_counts())



Grain Check: content_id is unique per row? True
Row Count: 5000
Missing Values: 46
Availability: 4587 rows out of 5000 survive the complete data filter.

Content Age Window Distribution:
content_age_window
180+ Days (Old)         2898
91-180 Days (Medium)    2019
31-90 Days (Recent)       83
Name: count, dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Ensure no NaNs for the ML test
df_clean = survivors.copy()

# THE TRAP: Add ONE label-derived column on purpose
# Simulating a metric that accidentally correlates 95% with our label
df_clean['leaky_metric'] = df_clean[label] * 0.95 + np.random.normal(0, 0.05, len(df_clean))

X_trap = df_clean[features + ['leaky_metric']]
y = df_clean[label]

# Watch the quick score jump toward perfect
model_trap = LogisticRegression(max_iter=1000)
model_trap.fit(X_trap, y)
trap_score = accuracy_score(y, model_trap.predict(X_trap))
print(f"Model score WITH the leaky feature (The Trap): {trap_score:.4f} (Dangerously perfect)")

# THE FIX: Delete the leakage and keep the honest number
X_honest = df_clean[features]
model_honest = LogisticRegression(max_iter=1000)
model_honest.fit(X_honest, y)
honest_score = accuracy_score(y, model_honest.predict(X_honest))
print(f"Model score AFTER deleting the leakage (Honest): {honest_score:.4f}")

Model score WITH the leaky feature (The Trap): 1.0000 (Dangerously perfect)
Model score AFTER deleting the leakage (Honest): 0.8116


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.